In [2]:
# Data_processor
import os
import re
import logging
from re import Pattern
from argparse import ArgumentParser, Namespace

In [35]:
DEFAULT_DATA_PATH: str = '/home/jovyan/llm-ap-generation/resources/data/CVE-PDDL'
DOMAIN_FILE: str = 'domain.pddl'
PROBLEM_FILE: str = 'problem.pddl'
AP_PATTERN: Pattern[str] = re.compile(r'AP\d+')
    

def standardize_simple_blocks(lines: list[str]) -> list[str]:
    """Standardize indentation for simple blocks like (:requirements, :types, etc.)"""
    new_lines = []
    indent_level = 0
    in_simple_block = False
    i = 0

    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped:
            i += 1
            continue

        simple_block_keywords = ['(:requirements', '(:types', '(:constants', '(:objects', '(:predicates', '(:init']
        if any(stripped.startswith(kw) for kw in simple_block_keywords):
            new_lines.append('  ' * indent_level + stripped + '\n')
            indent_level += 1
            in_simple_block = True
            i += 1
            continue

        if in_simple_block and stripped == ')':
            indent_level = max(0, indent_level - 1)
            new_lines.append('  ' * indent_level + stripped + '\n')
            in_simple_block = False
            i += 1
            continue

        if in_simple_block:
            new_lines.append('  ' * indent_level + stripped + '\n')
            i += 1
            continue

        # regular lines: parenthesis counting
        if stripped.startswith(')') and stripped.count('(') == 0:
            indent_level = max(0, indent_level - 1)
            new_lines.append('  ' * indent_level + stripped + '\n')
            i += 1
            continue

        new_lines.append('  ' * indent_level + stripped + '\n')
        net = stripped.count('(') - stripped.count(')')
        indent_level = max(0, indent_level + net)
        i += 1

    return new_lines


def format_predicate_block(lines: list[str], i: int, base_indent: str) -> tuple[list[str], int]:
    """Recursively format predicate blocks with nested (or (and ...) handling."""
    new_lines = []
    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped:
            i += 1
            continue
        # stop at next keyword or end of action
        if (stripped.startswith(':') or
                stripped.startswith('(:action') or
                stripped == ')'):
            break
        # nested (or or standalone (and spanning multiple lines
        if stripped in ('(or', '(and'):
            new_lines.append(base_indent + stripped + '\n')
            i += 1
            nested_lines, i = format_predicate_block(lines, i, base_indent + '  ')
            new_lines.extend(nested_lines)
            if i < len(lines) and lines[i].strip() == ')':
                new_lines.append(base_indent + ')' + '\n')
                i += 1
        else:
            new_lines.append(base_indent + stripped + '\n')
            i += 1
    return new_lines, i


def standardize_action_blocks(lines: list[str]) -> list[str]:
    """Standardize indentation for (:action blocks."""
    new_lines = []
    in_action_block = False
    action_indent = ''
    i = 0

    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped:
            i += 1
            continue

        # detect (:action block start
        if stripped.startswith('(:action'):
            in_action_block = True
            action_indent = ' ' * (len(lines[i]) - len(lines[i].lstrip()))
            new_lines.append(action_indent + stripped + '\n')
            i += 1
            continue

        if in_action_block:
            keyword_indent = action_indent + '  '

            # end of action block
            if stripped == ')':
                in_action_block = False
                new_lines.append(action_indent + stripped + '\n')
                i += 1
                continue

            # :parameters
            if stripped.startswith(':parameters'):
                new_lines.append(keyword_indent + stripped + '\n')
                i += 1
                continue

            # :precondition or :effect
            if stripped.startswith(':precondition') or stripped.startswith(':effect'):
                if '(and' in stripped:
                    keyword_part = stripped[:stripped.index('(and') + len('(and')]
                    new_lines.append(keyword_indent + keyword_part + '\n')
                    predicate_indent = keyword_indent + '  '
                    i += 1
                    predicate_lines, i = format_predicate_block(lines, i, predicate_indent)
                    new_lines.extend(predicate_lines)
                else:
                    new_lines.append(keyword_indent + stripped + '\n')
                    i += 1
                continue

            # other lines inside action
            new_lines.append(keyword_indent + stripped + '\n')
            i += 1
            continue

        # non-action lines: pass through unchanged
        new_lines.append(lines[i])
        i += 1

    return new_lines


def standardize_indentation(lines: list[str]) -> list[str]:
    """Main indentation function: remove inline comments, handle simple blocks, then action blocks."""
    # lines = standardize_simple_blocks(lines)
    lines = standardize_action_blocks(lines)
    return lines


def standardize_pddl(lines: list[str], filename: str = None, is_problem: bool = False) -> list[str]:
    new_lines = []
    for i, line in enumerate(lines):
        # if line.lstrip().startswith(';'):
        #      continue
        # if line.lstrip().startswith(';#'):
        #     continue
        # if line.lstrip().startswith(';@'):
        #     continue
        if line.strip() == '':
            continue
        if ';;' in line:
            line = line[:line.index(';;')].rstrip() + '\n'
        if ';' in line:
            line = line[:line.index(';')].rstrip() + '\n'
        new_lines.append(line)
    return new_lines


def process_problem(path: str, output_path: str):
    os.makedirs(output_path, exist_ok=True)
    for pddl_file, is_problem in [(DOMAIN_FILE, False), (PROBLEM_FILE, True)]:
        input_file = os.path.join(path, pddl_file)
        if not os.path.exists(input_file):
            logging.warning(f'File not found, skipping: {input_file}')
            continue
        with open(input_file, 'r') as f:
            lines = f.readlines()
        filename = os.path.splitext(pddl_file)[0]
        # 1. remove commented lines, empty line, ...
        # new_lines = standardize_pddl(lines, filename=filename, is_problem=is_problem)
        # 2. standardize intentation and parentheses
        # new_lines = standardize_indentation(lines)
        output_file = os.path.join(output_path, pddl_file)
        with open(output_file, 'w') as f:
            f.writelines(new_lines)
        logging.info(f'Written: {output_file}')

def process_data(path: str, output_path: str | None):
    if output_path is None:
        output_path = path
    for cve in os.listdir(path):
        cve_path = os.path.join(path, cve)
        if not os.path.isdir(cve_path):
            continue
        for ap in os.listdir(cve_path):
            if AP_PATTERN.match(ap):
                src = os.path.join(path, cve, ap)
                dst = os.path.join(output_path, cve, ap)
                logging.info(f'Processing: {src}')
                process_problem(src, dst)

def parse_arguments() -> Namespace:
    parser = ArgumentParser(description='Command line tool to preprocess PDDL files')
    parser.add_argument(
        '--data_path',
        type=str,
        default=DEFAULT_DATA_PATH,
        help='Path to the directory with the PDDL files'
    )
    parser.add_argument(
        '--output_path',
        type=str,
        default=None,
        help='Path to the output directory (if not provided files will be overwritten)'
    )
    return parser.parse_args()

def main():
    logging.basicConfig(level=logging.INFO)
    process_data(DEFAULT_DATA_PATH, None)
    return 0

if __name__ == '__main__':
    main()

INFO:root:Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP1
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP1/domain.pddl
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP1/problem.pddl
INFO:root:Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP2
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP2/domain.pddl
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP2/problem.pddl
INFO:root:Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP5
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP5/domain.pddl
INFO:root:Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CVE-2024-22262/AP5/problem.pddl
INFO:root:Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL/CV

In [59]:
import os
import re
import logging
from argparse import ArgumentParser, Namespace
from typing import Pattern

DEFAULT_DATA_PATH: str = '/home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL'
DOMAIN_FILE: str = 'domain.pddl'
PROBLEM_FILE: str = 'problem.pddl'
AP_PATTERN: Pattern[str] = re.compile(r'AP\d+')


def tokenize_pddl(text: str) -> list[str]:
    tokens = []
    i = 0
    while i < len(text):
        c = text[i]
        if c.isspace():
            i += 1
        elif c == ';':
            while i < len(text) and text[i] != '\n':
                i += 1
        elif c in '()':
            tokens.append(c)
            i += 1
        else:
            j = i + 1
            while j < len(text) and (not text[j].isspace()) and text[j] not in '();':
                j += 1
            tokens.append(text[i:j])
            i = j
    return tokens


def split_top_level_blocks(text: str) -> list[str]:
    """
    Split top-level blocks in domain.pddl and preserve the original text of each block.
    Example:
      (define ...)
        (:requirements ...)
        (:types ...)
        (:predicates ...)
        (:action ...)
    """
    lines = text.splitlines(keepends=True)
    blocks = []

    current = []
    depth = 0
    in_define_header = False
    started = False

    for line in lines:
        stripped = line.strip()

        if not started:
            if stripped.startswith('(define'):
                blocks.append(line)
                started = True
            continue

        # Keep "(domain xxx)" inside define header separately
        if stripped.startswith('(domain ') or stripped.startswith('(:domain '):
            blocks.append(line)
            continue

        if stripped == ')' and depth == 0:
            blocks.append(line)
            continue

        # Collect top-level sections
        if not current and stripped == '':
            continue

        open_count = line.count('(')
        close_count = line.count(')')

        if not current and stripped.startswith('(:'):
            current = [line]
            depth = open_count - close_count
            if depth <= 0:
                blocks.append(''.join(current))
                current = []
                depth = 0
            continue

        if current:
            current.append(line)
            depth += open_count - close_count
            if depth <= 0:
                blocks.append(''.join(current))
                current = []
                depth = 0
            continue

        # Preserve other scattered content as-is
        blocks.append(line)

    return blocks


def get_block_head(block_text: str) -> str | None:
    stripped = block_text.lstrip()
    if not stripped.startswith('(:'):
        return None

    m = re.match(r'\(\s*(:[^\s()]+)', stripped)
    if not m:
        return None
    return m.group(1)
    
def parse_one_sexp(tokens: list[str], pos: int = 0):
    if pos >= len(tokens):
        return None, pos

    if tokens[pos] == '(':
        items = []
        pos += 1
        while pos < len(tokens) and tokens[pos] != ')':
            item, pos = parse_one_sexp(tokens, pos)
            if item is not None:
                items.append(item)
        if pos < len(tokens) and tokens[pos] == ')':
            pos += 1
        return items, pos

    return tokens[pos], pos + 1


def sexp_to_str(sexp) -> str:
    if isinstance(sexp, str):
        return sexp
    return '(' + ' '.join(sexp_to_str(x) for x in sexp) + ')'


def format_types_like_block(sexp, indent: str = '  ') -> str:
    """
    Used for :types / :constants / :objects

    Rules:
    1. Each standalone symbol occupies one line
    2. Groups with '-' (inheritance/type declaration) stay on one line
       Example:
         exploit-technique - technique
         ?x ?y - actor
    """
    if not sexp:
        return ''

    keyword = sexp[0] if str(sexp[0]).startswith(':') else ':' + str(sexp[0])
    result = indent + f'({keyword}\n'
    child_indent = indent + '  '

    i = 1
    n = len(sexp)

    while i < n:
        item = sexp[i]

        # Nested structure → output directly as a single line
        if not isinstance(item, str):
            result += child_indent + sexp_to_str(item) + '\n'
            i += 1
            continue

        # Case 1: "name - parent"
        if i + 2 < n and sexp[i + 1] == '-' and isinstance(sexp[i + 2], str):
            result += child_indent + f'{sexp[i]} - {sexp[i + 2]}\n'
            i += 3
            continue

        # Case 2: multiple names share a type, e.g. a b c - t
        j = i
        names = []
        while j < n and isinstance(sexp[j], str):
            if sexp[j] == '-':
                break

            if j + 1 < n and sexp[j + 1] == '-':
                names.append(sexp[j])
                if j + 2 < n and isinstance(sexp[j + 2], str):
                    result += child_indent + ' '.join(names) + f' - {sexp[j + 2]}\n'
                    i = j + 3
                    break
                else:
                    result += child_indent + ' '.join(names) + '\n'
                    i = j + 1
                    break

            names.append(sexp[j])
            j += 1
        else:
            # No "-", so each is an independent type → one per line
            for name in names:
                result += child_indent + f'{name}\n'
            i = j
            continue

        if i != j:
            continue

        # Fallback: current token on its own line
        result += child_indent + f'{item}\n'
        i += 1

    result += indent + ')\n'
    return result


def format_predicates_block(sexp, indent: str = '  ') -> str:
    result = indent + '(:predicates\n'
    child_indent = indent + '  '
    for child in sexp[1:]:
        result += child_indent + sexp_to_str(child) + '\n'
    result += indent + ')\n'
    return result


def format_functions_block(sexp, indent: str = '  ') -> str:
    result = indent + '(:functions\n'
    child_indent = indent + '  '
    for child in sexp[1:]:
        result += child_indent + sexp_to_str(child) + '\n'
    result += indent + ')\n'
    return result


def format_logic_expr_compact(sexp, indent: str = '', max_inline_len: int = 160) -> str:
    """
    Compact logical expression formatter:
    - Atomic expressions in one line
    - and/or inline when possible
    - If multiline is needed, keep proper indentation
    """
    if isinstance(sexp, str):
        return indent + sexp

    if not sexp:
        return indent + '()'

    single = sexp_to_str(sexp)
    if len(indent) + len(single) <= max_inline_len:
        return indent + single

    head = sexp[0] if isinstance(sexp[0], str) else None

    if head in ('and', 'or'):
        child_indent = indent + '  '
        child_strs = [format_logic_expr_compact(child, '', max_inline_len).strip() for child in sexp[1:]]

        result = indent + f'({head}'
        if child_strs:
            result += ' ' + child_strs[0]
            for child in child_strs[1:]:
                result += '\n' + child_indent + child
        result += ')'
        return result

    return indent + single


def format_and_or_block_inline_first(
    keyword: str,
    expr,
    kw_indent: str = '    ',
    max_inline_len: int = 220
) -> str:
    """
    Format :precondition/:effect block.
    - (or (and ...) (and ...) ...) → first (and) inline with (or, rest aligned
    - other and/or → expanded multiline with alignment
    """
    if not isinstance(expr, list) or not expr:
        return kw_indent + f'{keyword} {sexp_to_str(expr)}\n'

    head = expr[0] if isinstance(expr[0], str) else None
    if head not in ('and', 'or'):
        return kw_indent + f'{keyword} {sexp_to_str(expr)}\n'

    children = expr[1:]
    if not children:
        return kw_indent + f'{keyword} ({head})\n'

    first_prefix = f'{kw_indent}{keyword} ({head} '
    aligned_indent = ' ' * len(first_prefix)

    lines_out = []
    first_child_str = format_nested_or_and(children[0], '')
    lines_out.append(first_prefix + first_child_str)

    for child in children[1:]:
        child_str = format_nested_or_and(child, aligned_indent)
        lines_out.append(child_str)

    lines_out[-1] = lines_out[-1].rstrip() + ')'
    return '\n'.join(lines_out) + '\n'


def format_nested_or_and(expr, indent: str) -> str:
    """
    Format a single child expression inside (or ...) or (and ...).
    If it is (or (and...) (and...)...), first (and) inline, rest aligned.
    Otherwise inline.
    """
    if isinstance(expr, str):
        return indent + expr

    head = expr[0] if isinstance(expr[0], str) else None

    if head in ('and', 'or') and len(expr) > 1:
        children = expr[1:]
        first_prefix = indent + f'({head} '
        aligned_indent = ' ' * len(first_prefix)

        lines_out = []
        lines_out.append(first_prefix + sexp_to_str(children[0]))
        for child in children[1:]:
            lines_out.append(aligned_indent + sexp_to_str(child))
        lines_out[-1] = lines_out[-1].rstrip() + ')'
        return '\n'.join(lines_out)

    return indent + sexp_to_str(expr)


def format_action_sexp(sexp, indent: str = '  ') -> str:
    if not isinstance(sexp, list) or len(sexp) < 2 or sexp[0] != ':action':
        return indent + sexp_to_str(sexp) + '\n'

    result = indent + f'(:action {sexp[1]}\n'
    kw_indent = indent + '  '

    i = 2
    while i < len(sexp):
        item = sexp[i]

        if isinstance(item, str) and item.startswith(':'):
            keyword = item
            i += 1

            if i >= len(sexp):
                result += kw_indent + keyword + '\n'
                break

            value = sexp[i]
            i += 1

            if keyword == ':parameters':
                result += kw_indent + f'{keyword} {sexp_to_str(value)}\n'
            elif keyword in (':precondition', ':effect'):
                result += format_and_or_block_inline_first(keyword, value, kw_indent)
            else:
                result += kw_indent + f'{keyword} {sexp_to_str(value)}\n'
        else:
            result += kw_indent + sexp_to_str(item) + '\n'
            i += 1

    result += indent + ')\n'
    return result


def standardize_domain_full(text: str) -> str:
    """
    Only reorder :action blocks;
    keep :types / :constants / :objects / :requirements / :predicates / :functions unchanged.
    """
    blocks = split_top_level_blocks(text)
    out = []

    for block in blocks:
        head = get_block_head(block)

        if head == ':action':
            tokens = tokenize_pddl(block)
            if tokens:
                sexp, _ = parse_one_sexp(tokens, 0)
                if sexp:
                    out.append(format_action_sexp(sexp))
                    continue

        # Keep non-action blocks unchanged
        out.append(block)

    return ''.join(out)


def strip_pddl_comments(text: str) -> str:                                                                                                                                                              
      """Remove comment-only lines, inline comments, and blank lines from PDDL text."""                                                                                                                   
      new_lines = []                                                                                                                                                                                      
      for line in text.splitlines(keepends=True):                                                                                                                                                         
          if line.strip() == '':                                                                                                                                                                          
              continue                                                                                                                                                                                    
          if line.lstrip().startswith(';'):                                                                                                                                                               
              continue                                                                                                                                                                                    
          if ';' in line:                                                                                                                                                                                 
              line = line[:line.index(';')].rstrip() + '\n'                                                                                                                                               
          if line.strip():                                                                                                                                                                                
              new_lines.append(line)                                                                                                                                                                      
      return ''.join(new_lines) 


def process_problem(path: str, output_path: str):
    os.makedirs(output_path, exist_ok=True)

    for pddl_file, is_problem in [(DOMAIN_FILE, False), (PROBLEM_FILE, True)]:
        input_file = os.path.join(path, pddl_file)
        if not os.path.exists(input_file):
            logging.warning(f'File not found, skipping: {input_file}')
            continue

        with open(input_file, 'r') as f:
            text = f.read()
            
        text = strip_pddl_comments(text)   
        
        if pddl_file == DOMAIN_FILE:
            new_text = standardize_domain_full(text)
        else:
            lines = text.splitlines(keepends=True)
            new_text = ''.join(standardize_pddl(lines, filename='problem', is_problem=True))

        output_file = os.path.join(output_path, pddl_file)
        with open(output_file, 'w') as f:
            f.write(new_text)

        print(f'Written: {output_file}')


def process_data(path: str, output_path: str | None):
    if output_path is None:
        output_path = path

    items = [x for x in os.listdir(path) if os.path.isdir(os.path.join(path, x))]

    if any(AP_PATTERN.match(x) for x in items):
        for ap in items:
            if AP_PATTERN.match(ap):
                src = os.path.join(path, ap)
                dst = os.path.join(output_path, ap)
                print(f'Processing: {src}')
                process_problem(src, dst)
        return

    for cve in items:
        cve_path = os.path.join(path, cve)
        for ap in os.listdir(cve_path):
            if AP_PATTERN.match(ap):
                src = os.path.join(cve_path, ap)
                dst = os.path.join(output_path, cve, ap)
                print(f'Processing: {src}')
                process_problem(src, dst)


def parse_arguments(argv=None) -> Namespace:
    parser = ArgumentParser(description='Command line tool to preprocess PDDL files')
    parser.add_argument('--data_path', type=str, default=DEFAULT_DATA_PATH)
    parser.add_argument('--output_path', type=str, default=None)
    return parser.parse_args(argv)


def main():
    logging.basicConfig(level=logging.INFO)
    args = parse_arguments([])
    process_data(args.data_path, args.output_path)
    return 0


if __name__ == '__main__':
    main()

Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP1
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP1/domain.pddl
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP1/problem.pddl
Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP2
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP2/domain.pddl
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP2/problem.pddl
Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP5
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP5/domain.pddl
Written: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP5/problem.pddl
Processing: /home/jovyan/llm-ap-generation/resources/data/CVE-PDDL-NNL/CVE-2024-22262/AP3
Written: /home/jovyan/llm-ap-generation/res